# Memory & GPU · **Python side**

*Python / Colab track — analogue of `4_memory_and_gpu.jl`.*

Locality (numpy is **row-major**, the opposite of Julia), arithmetic intensity & **batching**, then
the **GPU with PyTorch**.

> For the GPU: Colab ▸ *Edit ▸ Notebook settings ▸ GPU*.

In [ ]:
import timeit

def best_time(fn, repeat=7):
    """Time per call (s), %timeit-style: auto-scale then min over `repeat` trials."""
    number = 1
    while timeit.timeit(fn, number=number) < 0.05:
        number *= 10
    return min(timeit.repeat(fn, number=number, repeat=repeat)) / number

def ms(t): return f"{t*1e3:.3f} ms"
def us(t): return f"{t*1e6:.2f} µs"


In [ ]:
import numpy as np

## 1. Locality: numpy is *row-major* (C-order)

Two elements of the same **row** are neighbours in memory (C-order) — **the opposite of Julia**
(column-major). We sum the same matrix along the same axis, but laid out in C order and then in
Fortran order: only the **layout** changes.

**Bet before you look:** the Julia twin of this experiment (a hand-written loop) measures a **12×**
gap between the two layouts. What will numpy show?

In [ ]:
N = 8000
A = np.random.rand(N, N)            # C-order (contiguous rows) by default
A_f = np.asfortranarray(A)          # same numbers, contiguous columns

# reduction along axis 0 (we walk down the columns):
t_c = best_time(lambda: A.sum(axis=0))     # C-order: columns are strided
t_f = best_time(lambda: A_f.sum(axis=0))   # F-order: columns are contiguous
print("sum axis=0, C-order :", ms(t_c))
print("sum axis=0, F-order :", ms(t_f))
print(f"ratio : {t_c/t_f:.2f}×")

**≈ 1×. The layout made no difference — and that is the lesson.**

numpy's reduction does not walk the axis in the order *you* wrote it: `nditer` inspects the strides
and **reorders the iteration into memory order**. Whatever axis you ask for, numpy reads the buffer
the fast way. It protected you.

A hand-written loop has no such luck — nobody reorders it. That is exactly why the Julia twin, which
writes the loop out by hand, measures **12×** where numpy measures 1×. The hardware penalty is
identical in both languages; numpy just hides it behind the library.

So to *see* the cost in Python, we need work numpy **cannot** reorder: a copy between two arrays
whose layouts disagree. The destination must be filled in C order while the source can only be read
in F order — someone has to pay for the transposition.

In [ ]:
outC = np.empty((N, N))                      # destination: C-order

t_same = best_time(lambda: np.copyto(outC, A))    # C ← C : layouts agree
t_diff = best_time(lambda: np.copyto(outC, A_f))  # C ← F : real transposition traffic
print("copyto  C ← C :", ms(t_same))
print("copyto  C ← F :", ms(t_diff))
print(f"ratio : {t_diff/t_same:.2f}×")

**≈ 14×**, in pure numpy, on the same bytes and the same number of elements. Only the *order* of
the memory accesses changed.

Two things to take away:

- Getting the direction wrong doesn't give a wrong result — just slower code. In numpy the
  contiguous axis by default is the **last** one (rows); in Julia it's the **first** (columns).
- **A library can hide the penalty; it cannot remove it.** numpy saved you on `sum(axis=0)` because
  someone wrote the reordering for you. The moment you step outside what the library can rearrange —
  a layout-mismatched copy here, a hand-written loop in Julia — the memory hierarchy sends the bill.

## 2. Arithmetic intensity: matrix-vector vs matrix-matrix

In [ ]:
n = 2000
M  = np.random.rand(n, n)
vv = np.random.rand(n)
M2 = np.random.rand(n, n)

print("matrix × vector :", ms(best_time(lambda: M @ vv)))   # ~2n² ops / n² data → memory-bound
print("matrix × matrix :", ms(best_time(lambda: M @ M2)))   # ~2n³ ops / n² data → compute-bound

- **mat × vector**: ~2 ops per datum loaded → we wait on memory (*memory-bound*).
- **mat × matrix**: ~n ops per datum → we saturate the compute (*compute-bound*, BLAS level 3).

## 3. Batching: throughput **per example** (numpy)

One "layer" `W` (n→n) applied to a batch of `B` examples stacked as columns (`W @ X`).

In [ ]:
W = np.random.rand(n, n).astype(np.float32)
print("batch B | total time (ms) | per example (µs) | examples/s")
print("-"*60)
for B in (1, 8, 64, 256, 1024):
    X = np.random.rand(n, B).astype(np.float32)
    t = best_time(lambda: W @ X)
    print(f"{B:6d}  | {t*1e3:12.2f} | {t/B*1e6:14.2f} | {B/t:12.0f}")

The **total** time grows with `B`, but the time **per example** *drops*: `W` is loaded once and
serves the whole batch. That is "matrix-vector (memory-bound) → matrix-matrix (compute-bound)".

## 4. GPU with PyTorch

`torch` picks CPU or CUDA from the *device* of the tensors — just as Julia's multiple dispatch
picks the GPU version for `CuArray`s. Same operations, different hardware.

In [ ]:
import torch
gpu = torch.cuda.is_available()
print("CUDA available :", gpu, "|", torch.cuda.get_device_name(0) if gpu else "CPU only")

def bench_torch(fn, gpu, repeat=5):
    if gpu: torch.cuda.synchronize()
    # warm-up
    fn();  torch.cuda.synchronize() if gpu else None
    import time
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter(); fn()
        if gpu: torch.cuda.synchronize()
        best = min(best, time.perf_counter() - t0)
    return best

In [ ]:
# Matrix product CPU vs GPU
M = 4096
Ac = torch.rand(M, M); Bc = torch.rand(M, M)
print("matmul CPU :", ms(bench_torch(lambda: Ac @ Bc, gpu=False)))
if gpu:
    Ag = Ac.cuda(); Bg = Bc.cuda()
    print("matmul GPU :", ms(bench_torch(lambda: Ag @ Bg, gpu=True)))

In [ ]:
# Batching CPU vs GPU: throughput per example
if gpu:
    Wt = torch.rand(n, n, device="cuda")
    print("batch B | CPU µs/ex | GPU µs/ex | speedup")
    print("-"*48)
    Wc = Wt.cpu()
    for B in (1, 8, 64, 256, 1024):
        Xc = torch.rand(n, B); Xg = Xc.cuda()
        tc = bench_torch(lambda: Wc @ Xc, gpu=False)
        tg = bench_torch(lambda: Wt @ Xg, gpu=True)
        print(f"{B:6d}  | {tc/B*1e6:9.2f} | {tg/B*1e6:9.2f} | {tc/tg:10.1f}×")
else:
    print("(no GPU: enable the GPU accelerator in Colab)")

In [ ]:
# The red thread: Monte-Carlo π on GPU
def pi_torch(n, device):
    x = torch.rand(n, device=device); y = torch.rand(n, device=device)
    return 4.0 * torch.count_nonzero(x*x + y*y <= 1.0).item() / n

n_mc = 100_000_000
print("π (reference) :", np.pi)
print("π torch CPU :", pi_torch(n_mc, "cpu"))
if gpu:
    print("π torch GPU :", pi_torch(n_mc, "cuda"))
    print("CPU :", ms(bench_torch(lambda: pi_torch(n_mc, "cpu"), gpu=False)))
    print("GPU :", ms(bench_torch(lambda: pi_torch(n_mc, "cuda"), gpu=True)))

## Wrap-up — to compare with Julia/CUDA

| Idea | Python | Julia |
|---|---|---|
| memory order | **row-major** (C) | **column-major** |
| batching | `W @ X` numpy/torch | `W * X` |
| GPU | PyTorch (`.cuda()`) | CUDA.jl (`cu(...)`, dispatch) |

> Same story: respect **locality**, **batch** to move from memory-bound to compute-bound, and
> **feed** the GPU (only worth it at scale). Compare the GPU/CPU factor you get here with the one
> from the Julia notebook.